# Twitter Prediction Scorer — Colab Runner

Mimics the real `main.py` daemon end-to-end:
- Polls MongoDB every 5 seconds for unscored prediction records
- Runs the 8-step scoring pipeline (fetch prices → score → write back)
- Logs every step to stdout in the same format as production

**Run cells top-to-bottom, then leave the scorer cell running.**

---
### Before you start — add Colab Secrets
Go to the 🔑 **Secrets** tab (left sidebar) and add:

| Secret name | Value |
|---|---|
| `GITHUB_REPO` | `https://github.com/your-username/your-repo.git` |
| `MONGO_URI` | `mongodb://78.138.0.218:27017` |
| `MONGO_DB` | your database name |
| `MONGO_COLLECTION` | your collection name |
| `OPENAI_API_KEY` | your OpenAI key |
| `AZURE_SB_CONN_STR` | *(optional)* Azure Service Bus connection string |

Toggle **"Notebook access"** ON for each secret.

## Step 0 — Clone repo & register lib on the Python path

This is required because `main.ipynb` imports from `src/lib/utlis.py`.
Opening just the notebook file from GitHub does **not** bring `lib/` with it —
the full repo must be cloned so the import can resolve.

To pick up new code after a `git push`, re-run **only this cell** (no restart needed).

In [ ]:
import os, sys

REPO_DIR = "/content/repo"

try:
    from google.colab import userdata
    GITHUB_REPO = userdata.get('GITHUB_REPO')
except Exception:
    # Local fallback — set this manually if not in Colab
    GITHUB_REPO = os.environ.get('GITHUB_REPO', '')

if not GITHUB_REPO:
    raise ValueError("Add GITHUB_REPO to Colab Secrets (e.g. https://github.com/user/repo.git)")

if not os.path.exists(REPO_DIR):
    # First run — clone the repo
    print(f"Cloning {GITHUB_REPO} ...")
    result = os.system(f"git clone {GITHUB_REPO} {REPO_DIR}")
    if result != 0:
        raise RuntimeError("git clone failed — check that GITHUB_REPO is correct and the repo is public (or use a token).")
    print("✅ Repo cloned.")
else:
    # Subsequent runs or after a git push — pull latest changes
    print("Repo already cloned. Pulling latest changes ...")
    os.system(f"git -C {REPO_DIR} pull origin main")
    print("✅ Repo up to date.")

# Add the repo root to Python's module search path so that
# `from src.lib.utlis import timeasit` resolves correctly.
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"\nRepo structure:")
os.system(f"find {REPO_DIR} -not -path '*/.git/*' | sort")

## Step 1 — Install dependencies

In [ ]:
%pip install -q \
    pymongo==4.17.0 \
    yfinance==1.3.0 \
    openai==2.32.0 \
    pandas==3.0.2 \
    numpy==2.4.4 \
    pytz==2026.1.post1 \
    python-dotenv==1.2.2

print('✅ All packages installed.')

## Step 2 — Load secrets into environment variables

In [ ]:
import os

try:
    from google.colab import userdata

    os.environ['MONGO_URI']        = userdata.get('MONGO_URI')
    os.environ['MONGO_DB']         = userdata.get('MONGO_DB')
    os.environ['MONGO_COLLECTION'] = userdata.get('MONGO_COLLECTION')
    os.environ['OPENAI_API_KEY']   = userdata.get('OPENAI_API_KEY')

    try:
        os.environ['AZURE_SB_CONN_STR'] = userdata.get('AZURE_SB_CONN_STR')
    except Exception:
        pass

    print('✅ Secrets loaded from Colab Secrets tab.')

except ModuleNotFoundError:
    from dotenv import load_dotenv
    load_dotenv()
    print('✅ Secrets loaded from local .env file.')

required = ['MONGO_URI', 'MONGO_DB', 'MONGO_COLLECTION', 'OPENAI_API_KEY']
missing  = [k for k in required if not os.environ.get(k)]
if missing:
    print(f'⚠️  Missing secrets: {missing}')
else:
    print('✅ All required secrets present.')

## Step 3 — Verify MongoDB connection

In [ ]:
import os
from pymongo import MongoClient
from pymongo.errors import ConnectionFailure

MONGO_URI        = os.environ['MONGO_URI']
MONGO_DB         = os.environ['MONGO_DB']
MONGO_COLLECTION = os.environ['MONGO_COLLECTION']

try:
    client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)
    client.admin.command('ping')
    db         = client[MONGO_DB]
    collection = db[MONGO_COLLECTION]
    count      = collection.count_documents({'scored': False})
    print(f'✅ MongoDB connected — {count} unscored record(s) waiting.')
except ConnectionFailure as e:
    print(f'❌ Cannot reach MongoDB: {e}')
    print('   Check MONGO_URI and that 78.138.0.218:27017 is reachable from Colab.')

## Step 4 — Run the scorer

> **This cell runs forever.** It polls MongoDB every 5 seconds.
> Stop it with the ⏹ button or `Kernel → Interrupt`.
> Logs print in real time below the cell.

In [ ]:
import os
import time
import logging
from datetime import datetime, timezone

import pandas as pd
import yfinance as yf
from openai import OpenAI
from pymongo import MongoClient

# Import helper from lib/ — works because Step 0 added the repo root to sys.path
from src.lib.utlis import timeasit

# ── Config ────────────────────────────────────────────────────────────────────
MONGO_URI        = os.environ['MONGO_URI']
MONGO_DB         = os.environ['MONGO_DB']
MONGO_COLLECTION = os.environ['MONGO_COLLECTION']
OPENAI_API_KEY   = os.environ['OPENAI_API_KEY']
POLL_INTERVAL    = 5
DATE_CUTOFF      = '2026-04-12'

# ── Clients ───────────────────────────────────────────────────────────────────
mongo  = MongoClient(MONGO_URI)
db     = mongo[MONGO_DB]
col    = db[MONGO_COLLECTION]
openai = OpenAI(api_key=OPENAI_API_KEY)

# ── Logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
log = logging.getLogger('scorer')

# ── Pipeline steps ────────────────────────────────────────────────────────────

@timeasit
def step1_fetch_record():
    return col.find_one(
        {'scored': False, 'posted_at': {'$gte': DATE_CUTOFF}},
        sort=[('posted_at', 1)]
    )


def step2_extract_prediction(record):
    return {
        'record_id':   str(record['_id']),
        'username':    record.get('username', 'unknown'),
        'ticker':      record.get('ticker', '').upper(),
        'direction':   record.get('direction', '').upper(),
        'posted_at':   record.get('posted_at'),
        'target_date': record.get('target_date'),
    }


def step3_fetch_prices(ticker, posted_at, target_date):
    return yf.Ticker(ticker).history(start=posted_at, end=target_date, interval='1d')


def step4_openai_evaluate(prediction, prices_df):
    import json
    price_summary = (
        prices_df[['Open', 'Close']].tail(5).to_string()
        if not prices_df.empty else 'No price data available.'
    )
    prompt = (
        f"Prediction: @{prediction['username']} predicted {prediction['ticker']} "
        f"would go {prediction['direction']} from {prediction['posted_at']} "
        f"to {prediction['target_date']}.\n"
        f"Price data:\n{price_summary}\n"
        f'Did the prediction succeed? Answer with JSON: {{"success": true|false, "reason": "one sentence"}}'
    )
    response = openai.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        response_format={'type': 'json_object'},
        max_tokens=100,
    )
    return json.loads(response.choices[0].message.content)


def step5_batch_stats(username, success):
    pipeline = [
        {'$match': {'username': username, 'scored': True}},
        {'$group': {
            '_id': None,
            'wins':  {'$sum': {'$cond': ['$score_result.success', 1, 0]}},
            'total': {'$sum': 1},
        }}
    ]
    rows = list(col.aggregate(pipeline))
    wins  = (rows[0]['wins'] if rows else 0) + (1 if success else 0)
    total = (rows[0]['total'] if rows else 0) + 1
    return {'wins': wins, 'total': total, 'win_rate': round(wins / total, 4)}


def step6_user_score(username, batch):
    return {
        'username':   username,
        'wins':       batch['wins'],
        'total':      batch['total'],
        'win_rate':   batch['win_rate'],
        'updated_at': datetime.now(timezone.utc).isoformat(),
    }


def step7_save_results(record_id, openai_output, user_score, batch):
    from bson import ObjectId
    col.update_one(
        {'_id': ObjectId(record_id)},
        {'$set': {'score_result': openai_output, 'batch_stats': batch, 'user_score': user_score}}
    )


def step8_mark_scored(record_id):
    from bson import ObjectId
    col.update_one(
        {'_id': ObjectId(record_id)},
        {'$set': {'scored': True, 'scored_at': datetime.now(timezone.utc).isoformat()}}
    )


# ── Main loop ─────────────────────────────────────────────────────────────────

def run_pipeline(record):
    overall_start = time.time()

    t = time.time(); log.info(f'[SCORE] Step 1 (Get raw data) took {time.time()-t:.2f}s')

    t = time.time(); pred = step2_extract_prediction(record)
    log.info(f'[SCORE] Step 2 (Get predictions) took {time.time()-t:.2f}s')

    t = time.time(); prices = step3_fetch_prices(pred['ticker'], pred['posted_at'], pred['target_date'])
    log.info(f'[SCORE] Step 3 (Prediction success) took {time.time()-t:.2f}s')

    t = time.time(); openai_out = step4_openai_evaluate(pred, prices)
    log.info(f'[SCORE] Step 4 (Batches) took {time.time()-t:.2f}s')

    t = time.time(); batch = step5_batch_stats(pred['username'], openai_out.get('success', False))
    log.info(f'[SCORE] Step 5 (Scores) took {time.time()-t:.2f}s')

    t = time.time(); user_score = step6_user_score(pred['username'], batch)
    log.info(f'[SCORE] Step 6 (Save API output) took {time.time()-t:.2f}s')

    t = time.time(); step7_save_results(pred['record_id'], openai_out, user_score, batch)
    log.info(f'[SCORE] Step 7 (Save scores) took {time.time()-t:.2f}s')

    t = time.time(); step8_mark_scored(pred['record_id'])
    log.info(f'[SCORE] Step 8 (Mark raw data scored) took {time.time()-t:.2f}s')

    log.info(
        f'[SCORE] ✅ Completed {pred["record_id"]} for user {pred["username"]} '
        f'[{pred["ticker"]} {pred["direction"]}] in {time.time()-overall_start:.2f}s'
    )
    return True


print('=' * 60)
print('  Twitter Prediction Scorer — Live Worker')
print(f'  DB: {MONGO_DB}.{MONGO_COLLECTION} @ {MONGO_URI}')
print(f'  Polling for unscored records (date >= {DATE_CUTOFF})')
print('=' * 60)

while True:
    try:
        record = step1_fetch_record()

        if not record:
            log.info(f'No unscored records found. Retrying in {POLL_INTERVAL}s ...')
            time.sleep(POLL_INTERVAL)
            continue

        record_id = str(record['_id'])
        log.info(
            f'Processing record {record_id} — @{record.get("username")} | '
            f'{record.get("ticker")} {record.get("direction")} | posted: {record.get("posted_at")}'
        )
        run_pipeline(record)
        log.info(f'Scored record {record_id} ✓')

    except KeyboardInterrupt:
        log.info('Scorer stopped by user.')
        break
    except Exception as e:
        log.error(f'Pipeline error: {e}', exc_info=True)
        time.sleep(POLL_INTERVAL)

---
## Optional — Demo mode (no real DB or API needed)

Run this cell **instead of** Step 4 if you want to show the system working
without live credentials. Uses fake records and simulated delays.
Step 0 (clone) is still required so `src/lib/utlis.py` is available.

In [ ]:
import time, random
from datetime import datetime
from src.lib.utlis import timeasit

FAKE_USERS      = ["coachnickmoney", "cryptoking", "traderpro99", "wallstreetbull", "moonshot_alex"]
FAKE_ASSETS     = ["BTC", "ETH", "TSLA", "AAPL", "SOL", "NVDA"]
FAKE_DIRECTIONS = ["LONG", "SHORT"]
POLL_INTERVAL   = 5
DATE_CUTOFF     = "2026-04-12"


@timeasit
def fake_score(record_id, username, asset, direction):
    print(f"[SCORE] Processing started for message ID: {record_id}")
    for step, name in [
        (random.uniform(0.2, 0.5), "Get raw data"),
        (random.uniform(0.3, 0.6), "Get predictions"),
        (random.uniform(0.2, 0.4), "Prediction success"),
        (random.uniform(0.3, 0.5), "Batches"),
        (random.uniform(0.2, 0.4), "Scores"),
        (random.uniform(0.1, 0.3), "Save API output"),
        (random.uniform(0.2, 0.4), "Save scores"),
        (random.uniform(0.1, 0.2), "Mark raw data scored"),
    ]:
        time.sleep(step)
        print(f"[SCORE] Step {FAKE_ASSETS.index(asset)+1 if asset in FAKE_ASSETS else '?'} ({name}) took {step:.2f}s")
    print(f"[SCORE] ✅ Completed {record_id} for user {username} [{asset} {direction}]")


print("=" * 60)
print("  Twitter Prediction Scorer — Demo Worker")
print(f"  Polling for unscored records (date >= {DATE_CUTOFF})")
print("=" * 60)

record_counter = 0
while True:
    if random.random() >= 0.70:
        print(f"No unscored records found. Retrying in {POLL_INTERVAL}s ...")
        time.sleep(POLL_INTERVAL)
        continue

    record_counter += 1
    record_id = f"rec_{datetime.now().strftime('%H%M%S')}_{record_counter:04d}"
    username  = random.choice(FAKE_USERS)
    asset     = random.choice(FAKE_ASSETS)
    direction = random.choice(FAKE_DIRECTIONS)
    posted    = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    print(f"Processing record {record_id} — @{username} | {asset} {direction} | posted: {posted}")
    try:
        fake_score(record_id, username, asset, direction)
        print(f"Scored record {record_id} ✓")
    except Exception as e:
        print(f"Failed to score record {record_id}: {e}")
        time.sleep(POLL_INTERVAL)